# jepa-as-a-tool: the second run (Muse Spark, Kaggle)

The first overnight run trained everything, but two bugs spoiled part of it. Both are fixed on `dev`:
- **3D evaluations with the world model crashed** (rendering from several threads): only 2 episodes
  per world ran. They run now, with the tabletop model the first run trained.
- **Billiards balls could fly off the table** (unstable cushions): its data, translator, transfer
  test and predictor are rebuilt on the fixed physics, and its evaluations run again.

Tabletop and PushT models are kept as they are: their data is not rebuilt.

| Step | What | GPU | Time (est.) |
|---|---|---|---|
| 1. Start | evaluations in the background: tabletop + new look with the world model, billiards alone and with the real physics; billiards data; the agent's PushT pushes | CPU + API | ~15 min, evaluations all night |
| 2. Latents | billiards + the agent's pushes | 2 × T4 | ~40 min |
| 3. Billiards translator | its own translator, then the transfer test | 1 GPU | ~15 min |
| 4. Billiards predictor | then its evaluation with the world model starts | 1 GPU | ~2 h |
| 5. Learning from use | v2 vs fine-tuned on the same held-out agent pushes | 1 GPU | ~5 min |
| 6. Finish | wait for the evaluations, reports, `overnight.zip` | — | until the last episode |

About 550 episodes in contributor mode.

**Before running (Kaggle):**
1. Settings: Accelerator **GPU T4 x2**, Internet **on**.
2. Add-ons → Secrets → `META_API_KEY`, ticked for this notebook.
3. Add Input: the **`overnight-inputs`** dataset (as last time), and the **output of the first run**
   (Add Input → Notebooks → your overnight notebook, its latest version).
4. **Save Version → Save & Run All**: it keeps running with the browser closed.

Every step skips what is already done. If this run stops, attach **this** notebook's output as an
input of a new version: it continues (the old billiards is deleted only once).

In [ ]:
import os
import sys

if os.path.isdir("/kaggle/working"):
    WORK = "/kaggle/working"
elif os.path.isdir("/content"):
    WORK = "/content"
else:
    WORK = os.getcwd()
REPO = f"{WORK}/jepa-as-a-tool"
if os.path.isdir(REPO):
    !git -C {REPO} pull
else:
    !git clone -b dev https://github.com/RahulCHANDER25/jepa-as-a-tool.git {REPO}
%cd {REPO}
!pip install -q -e .

# `pip install -e` only reaches new Python processes (the pipeline's jobs). This kernel started
# before it, so it finds `jat` through the source folder: same code as the jobs.
if f"{REPO}/src" not in sys.path:
    sys.path.insert(0, f"{REPO}/src")
!git log --oneline -1

## 0. Machine check and preflight

Rendering, then one job per GPU exactly like the pipeline's: CUDA, both 3D worlds, and rendering
from several threads at once (the bug that crashed the evaluations). If a line says FAILED, stop and
send it to me.

In [ ]:
import subprocess
import sys

!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2
!df -h /tmp {WORK} | tail -2

CHECK = "from jat.worlds import get_world; f = get_world('billiards').render([150, 256, 400, 100]); print('render ok', f.shape, f.mean().round(1))"


def renders(gl):
    env = {**os.environ, "MUJOCO_GL": gl}
    out = subprocess.run([sys.executable, "-c", CHECK], env=env, capture_output=True, text=True)
    print(gl, "->", (out.stdout or out.stderr).strip().splitlines()[-1])
    return out.returncode == 0


if renders("egl"):
    os.environ["MUJOCO_GL"] = "egl"
else:
    !apt-get -qq install -y libosmesa6 > /dev/null
    assert renders("osmesa"), "MuJoCo cannot render here"
    os.environ["MUJOCO_GL"] = "osmesa"
    os.environ["PYOPENGL_PLATFORM"] = "osmesa"
print("MUJOCO_GL =", os.environ["MUJOCO_GL"])

In [ ]:
import torch

from jat.pipeline import preflight

GPUS = torch.cuda.device_count()
preflight(GPUS)

## 1. API key and model

In [ ]:
def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except ImportError:
        pass
    try:
        from google.colab import userdata

        return userdata.get(name)
    except ImportError:
        pass
    from getpass import getpass

    return getpass(f"{name}: ")


os.environ["META_API_KEY"] = secret("META_API_KEY")
os.environ["META_API_BASE_URL"] = "https://api.meta.ai/v1"
# Contributor mode: same model, ~10x cheaper; Meta may train on these requests.
os.environ["META_MODEL"] = "muse-spark-1.3-contributor"
print("key loaded:", bool(os.environ["META_API_KEY"]), "| model:", os.environ["META_MODEL"])

## 2. Inputs

`wm-v2/` and `eval-wm-v2/` from `overnight-inputs`, and the first run's output (`overnight/`). If an
earlier version of **this** notebook is attached too, it wins: its progress is kept.

In [ ]:
import glob
import shutil
import zipfile
from pathlib import Path

INPUTS = Path(WORK) / "inputs"
ROOTS = ["/kaggle/input", str(INPUTS)]


def search(pattern):
    return sorted(Path(p) for r in ROOTS for p in glob.glob(f"{r}/**/{pattern}", recursive=True))


for archive in search("*.zip"):
    if archive.name.startswith(("overnight-inputs", "wm-v2", "eval-wm-v2")):
        zipfile.ZipFile(archive).extractall(INPUTS)

WM_V2 = next((p.parent for p in search("wm-v2/predictor.pt")), None)
EVAL_V2 = next((p.parent for p in search("eval-wm-v2/episodes")), None)
print("wm-v2:     ", WM_V2 or "MISSING")
print("eval-wm-v2:", EVAL_V2 or "MISSING")
assert WM_V2 and EVAL_V2, "attach the overnight-inputs dataset: see 'Before running', step 3"

OUT = Path(WORK) / "overnight"
runs = [
    p.parents[2]
    for p in search("overnight/runs/tabletop/predictor.pt")
    if "/kaggle/input" in str(p)
]
# A version of this notebook (it has the marker) wins over the first run
previous = sorted(runs, key=lambda p: not (p / "runs" / "billiards-physics-2").exists())
if previous and not (OUT / "results").exists():
    for part in ("runs", "results", "logs"):
        if (previous[0] / part).exists():
            shutil.copytree(previous[0] / part, OUT / part)
    print("resumed from", previous[0])
print(
    len(list(OUT.glob("results/*/episodes/*.json"))),
    "episodes,",
    len(list(OUT.glob("runs/*/*.pt"))),
    "model files",
)
assert (OUT / "runs/tabletop/predictor.pt").exists(), "attach the first run's output: step 3"

## 3. The pipeline

Big data goes to `/tmp` (not saved); runs, results and logs go to `overnight/` in the output. Each
step's full log is in `overnight/logs/<step>.log`.

In [ ]:
from jat.pipeline import Pipeline

Path("/tmp/jat-data").mkdir(exist_ok=True)
if not (OUT / "data").exists():
    (OUT / "data").symlink_to("/tmp/jat-data")  # latents stay out of the saved output

P = Pipeline(
    OUT, size="full", wm_v2=WM_V2, eval_v2=EVAL_V2, gpus=GPUS, provider="meta", eval_workers=6
)

### After a fix is pushed (harmless otherwise)

Pulls the fix into the running pipeline without losing anything; then run the failed cell again.

In [ ]:
!git -C {REPO} pull
import importlib

import jat.pipeline

importlib.reload(jat.pipeline)
P.__class__ = jat.pipeline.Pipeline
print("pipeline code:", jat.pipeline.__file__)

## Step 1: start (old billiards deleted once, evaluations in the background, data)

`overnight/logs/eval-*.log` shows the evaluations' progress.

In [ ]:
P.reset_billiards()
P.stage_second_data()

## Step 2: V-JEPA 2 latents (2 GPUs, ~40 min)

In [ ]:
P.stage_encode()

## Step 3: billiards translator and the transfer test (~15 min)

Gate 2 for billiards: under ~5 px. Transfer: a head on the frozen PushT + tabletop trunk vs a
translator from scratch, per number of training episodes.

In [ ]:
summary = P.stage_billiards_translator()
summary["transfer"]

## Step 4: billiards predictor (~2 h)

Its evaluation with the world model starts as soon as it is saved.

In [ ]:
P.stage_predictors()

## Step 5: learning from use, v2 vs fine-tuned on the same agent pushes (~5 min)

In [ ]:
import json

P.stage_learning_check()
for tag in ("v2", "ft"):
    path = OUT / f"runs/pusht-ft/wm_eval_agent_{tag}.json"
    if path.exists():
        row = json.loads(path.read_text())["test"][1]
        print(
            f"{tag}: one push ahead, {row['px_median']:.1f} px, {row['deg_median']:.1f} deg median"
        )

## Step 6: finish

Waits for every evaluation (failed episodes get one retry), writes `report.txt` per world, and packs
`overnight.zip`: runs, results and logs, no datasets.

In [ ]:
archive = P.stage_finish()
!ls -lh {archive}
for report in sorted(OUT.glob("results/*/report.txt")):
    print(f"\n===== {report.parent.name}\n{report.read_text()}")

## Send back

Download `overnight/overnight.zip` (Output tab) and send it.